In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

N   = 50
IDX = 16

PROJECT_ROOT = Path.cwd().parent
HIST_DIR     = PROJECT_ROOT / "data" / "0" / f"N{N}" / "training_histories"
path = HIST_DIR / f"training_history_{IDX:04d}.json"
print("Cargando:", path)
with open(path) as f:
    hist = json.load(f)

print(f"N={hist['N']}  N_A={hist['N_A']}  T={hist['T']}  chunk_size={hist['chunk_size']}")
print(f"Fases: {hist['phases']}")
print(f"steps: {len(hist['steps'])}   dkl_records: {len(hist['dkl_records'])}")

In [ ]:
# ---- F(global_step) ----
steps  = hist["steps"]
gs     = np.array([s["global_step"] for s in steps])
F      = np.array([s["F"]           for s in steps])
phase  = np.array([s["phase"]       for s in steps])

# Fronteras de fase (índices de global_step donde cambia `phase`)
phase_edges = [gs[i] for i in range(1, len(phase)) if phase[i] != phase[i-1]]

# ---- DKL ----
dkl_recs = hist["dkl_records"]
dkl_gs   = np.array([r["global_step"] for r in dkl_recs])
dkl_val  = np.array([r["dkl"]         for r in dkl_recs])
dkl_tag  = np.array([r["tag"]         for r in dkl_recs])

print("Tags presentes:", sorted(set(dkl_tag.tolist())))
print(f"F_min = {F.min():.4f}   F_max = {F.max():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(gs, F, "-", lw=1.2, color="tab:blue", label=r"$F$")

phase_names = [p["method"] for p in hist["phases"]]
bounds = [gs[0]] + phase_edges + [gs[-1] + 1]
for i, (a, b) in enumerate(zip(bounds[:-1], bounds[1:])):
    ax.axvspan(a, b, alpha=0.06,
               color="tab:blue" if phase_names[i]=="swap" else "tab:red")
    ax.text((a + b) / 2, ax.get_ylim()[1], phase_names[i],
            ha="center", va="top", fontsize=10, alpha=0.7)
for b in phase_edges:
    ax.axvline(b, color="k", ls="--", alpha=0.3)

ax.set_xlabel("global step")
ax.set_ylabel(r"$F = E - T\,S_2$")
ax.set_title(f"N={hist['N']}, N_A={hist['N_A']}, T={hist['T']}   "
             f"F_min={F.min():.4f}   idx={IDX}")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

styles = {
    "monitor":           dict(marker="o", s=14, color="tab:orange", label="monitor"),
    "initial_before":    dict(marker="^", s=45, color="tab:purple", label="initial (pre)"),
    "initial_after":     dict(marker="v", s=45, color="tab:cyan",   label="initial (post)"),
    "warm_start_before": dict(marker="s", s=20, color="tab:red",    label="warm_start (pre)"),
    "warm_start_after":  dict(marker="s", s=20, color="tab:green",  label="warm_start (post)"),
}

for tag, st in styles.items():
    m = dkl_tag == tag
    if m.any():
        ax.scatter(dkl_gs[m], dkl_val[m],
                   marker=st["marker"], s=st["s"], color=st["color"],
                   label=st["label"], zorder=3, edgecolors="k", linewidths=0.3)

for b in phase_edges:
    ax.axvline(b, color="k", ls="--", alpha=0.3)

ax.set_yscale("log")
ax.set_xlabel("global step")
ax.set_ylabel(r"$D_{KL}(p_\theta \,\|\, p_R)$")
ax.set_title(f"Fidelidad de la red reversa  (idx={IDX})")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8, ncol=2, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
inner = [(r["inner_step"], r["dkl"]) for r in dkl_recs
         if r["tag"] == "initial_train_step"]
if inner:
    inner.sort()
    xs, ys = zip(*inner)
    ax.plot(xs, ys, "-", color="tab:purple", lw=1.2)
    ax.set_title(f"initial_train  (n_steps={len(xs)})")
    ax.set_xlabel("inner step")
    ax.set_ylabel(r"$D_{KL}$")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3, which="both")

ax = axes[1]
ws_events = {}
for r in dkl_recs:
    if r["tag"] == "warm_start_step":
        ws_events.setdefault(r["global_step"], []).append(
            (r["inner_step"], r["dkl"]))
for gstep, pts in ws_events.items():
    pts.sort()
    xs, ys = zip(*pts)
    ax.plot(xs, ys, "-", lw=1.0, alpha=0.7, label=f"@ step {gstep}")
ax.set_title(f"warm_start  ({len(ws_events)} eventos)")
ax.set_xlabel("inner step")
ax.set_ylabel(r"$D_{KL}$")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
if len(ws_events) <= 8:
    ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
INDICES = [10, 12, 14, 16, 18, 20, 22, 24, 26, 28]

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.viridis

for k, idx in enumerate(INDICES):
    p = HIST_DIR / f"training_history_{idx:04d}.json"
    if not p.exists():
        print("falta:", p.name); continue
    with open(p) as f:
        h = json.load(f)
    gs_ = np.array([s["global_step"] for s in h["steps"]])
    F_  = np.array([s["F"]           for s in h["steps"]])
    ax.plot(gs_, F_, lw=1.0, color=cmap(k / max(1, len(INDICES)-1)),
            label=f"idx={idx}  T={h['T']}")

ax.set_xlabel("global step")
ax.set_ylabel(r"$F$")
ax.set_title(f"N={N}  —  curvas de F por temperatura")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()